In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler, LabelEncoder


# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")
csv_path = os.path.join(path, "Q1_data.csv")
print("Path to dataset files:", path)


In [ ]:

# Task 1: Write your code here:
df = pd.read_csv(csv_path)


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery Time Distribution')
plt.xlabel('Delivery')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df_new = df.drop('Order_ID',axis=1).dropna()
df_new

In [ ]:
# Task 2: Write your code here:
def check_missing_values(df):

  # Get missing values using pandas
  missing_values = df_new.isnull().sum()

  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])

  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df_new)
print(df_new.isnull().sum())


In [ ]:
# Task 3: Write your code here:
print("Checking for duplicate rows...")
duplicate_rows = df_new.duplicated().sum()
if duplicate_rows > 0:
    print(f"Found {duplicate_rows} duplicate rows. Removing them...")
    df_new.drop_duplicates(inplace=True)
    print("Duplicate rows removed.")
else:
    print("No duplicate rows found.")


In [ ]:
# Task 4: Write your code here:
categorical_cols = df_new.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

from sklearn.preprocessing import LabelEncoder

label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  df_new[col] = le.fit_transform(df_new[col])
  label_encoders[col] = le

df_new

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

numerical_cols = df_new.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nScaled ranges - Min: {X_train_scaled.min():.2f}, Max: {X_train_scaled.max():.2f}")
pd.DataFrame(X_train_scaled, columns=X_train.columns).head(3)

In [ ]:
# Task 6: Write your code here:
def check_target_imbalance(df, target_column):
  print("Target Distribution:")

  df_new[target_column].hist()  # Yeah you can just do this :)
  plt.show()

check_target_imbalance(df, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
# # Define features (X) and target (y)


# Train-test split (80% train, 20% test)
X = df_new.drop("Delivery_Time", axis=1).astype(float)
y = df_new['Delivery_Time'].astype(float)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
X_train, X_test, y_train, y_test

In [ ]:
def gradient_descent(X, y, learning_rate, n_iters=500):
  m, n = X.shape  # m rows, n columns (dimensions)
  theta = np.zeros(n)  # initialize a zeros weight vector with n dimensions
  losses = []

  for _ in tqdm(range(n_iters), desc="Training Linear Regression"):
    y_hat = np.dot(X, theta)
    gradient = np.dot(X.T, (y_hat - y)) / m
    theta -= learning_rate * gradient

    loss = mean_squared_error(y, y_hat)
    losses.append(loss)

  return theta, losses

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
import numpy as np

# Initialize KFold cross-validation
# Using n_splits = 5 as a common practice
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Initialize lists to store MAE scores for each fold
mae_scores = []


rf_model = RandomForestRegressor(n_estimators=200)


scaler = StandardScaler()

print("Starting KFold Cross-Validation...")
for fold, (train_index, test_index) in enumerate(kf.split(X)):
    print(f" Fold {fold+1}/{kf.n_splits} ")

    X_train_fold, X_test_fold = X.iloc[train_index], X.iloc[test_index]
    y_train_fold, y_test_fold = y.iloc[train_index], y.iloc[test_index]

    X_train_scaled_fold = scaler.fit_transform(X_train_fold)
    X_test_scaled_fold = scaler.transform(X_test_fold)

    rf_model.fit(X_train_scaled_fold, y_train_fold)

    y_pred_fold = rf_model.predict(X_test_scaled_fold)

    mae = mean_absolute_error(y_test_fold, y_pred_fold)
    mae_scores.append(mae)
    print(f"MAE for Fold {fold+1}: {mae:.2f}")

average_mae = np.mean(mae_scores)
print(f"\nAverage MAE across all folds: {average_mae:.2f}")


model = rf_model
X_test_scaled = X_test_scaled_fold

In [ ]:
# Task 1: Write your code here:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:

y_pred = model.predict(X_test_scaled)

plt.figure(figsize=(10, 5))
plt.hist(y_pred, bins=50, edgecolor='black')
plt.title('Predicted Delivery Time Distribution')
plt.xlabel('Predicted Delivery Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task Bonus: Write your code here: